In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\Gopi.Battineni\\AppData\\Local\\anaconda3\\Lib\\site-packages\\sklearn\\_cyutility.cp310-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo


In [6]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

real_estate = fetch_ucirepo(id=477)
X = real_estate.data.features.copy()
y = real_estate.data.targets
print(real_estate.metadata)
print(real_estate.variables)

X = X.drop(columns=["X1 transaction date"], errors="ignore")
if "X4 number of convenience stores" in X.columns:
    X["X4 number of convenience stores"] = pd.to_numeric(
        X["X4 number of convenience stores"], errors="coerce"
    )

data = pd.concat([X, y], axis=1)
target_col = "Y house price of unit area"

# Drop date/time/session/ID features before generator training (high cardinality).
_drop_feature_cols = [
    "Date", "Time", "date_time",
    "session_id", "Session ID", "Session_ID", "session",
    "X1 transaction date",
]
data = data.drop(columns=[c for c in _drop_feature_cols if c in data.columns], errors="ignore")

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

# Experiment settings
FAST_MODE = True
RUN_QUALITY_EVAL = True
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

ALL_GENERATORS = [
    'CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'TabDDPM', 'ForestDiffusion'
]
GENERATORS_TO_EVAL = ALL_GENERATORS

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []

def align_to_train_schema(df, reference_df, label_col):
    """Map synthetic data to match the schema of the reference training data."""
    df = df.copy()
    y = pd.to_numeric(df[label_col], errors='coerce').fillna(0)
    X = df.drop(columns=[label_col], errors='ignore')
    X_ref = reference_df.drop(columns=[label_col], errors='ignore')

    if X.select_dtypes(include=['object', 'string', 'category']).shape[1] > 0:
        X = pd.get_dummies(X, drop_first=True)

    X = X.reindex(columns=X_ref.columns, fill_value=0)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

    out = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
    out.columns = reference_df.columns
    return out


{'uci_id': 477, 'name': 'Real Estate Valuation', 'repository_url': 'https://archive.ics.uci.edu/dataset/477/real+estate+valuation+data+set', 'data_url': 'https://archive.ics.uci.edu/static/public/477/data.csv', 'abstract': 'The real estate valuation is a regression problem. The market historical data set of real estate valuation are collected from Sindian Dist., New Taipei City, Taiwan. ', 'area': 'Business', 'tasks': ['Regression'], 'characteristics': ['Multivariate'], 'num_instances': 414, 'num_features': 6, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Y house price of unit area'], 'index_col': ['No'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2018, 'last_updated': 'Mon Feb 26 2024', 'dataset_doi': '10.24432/C5J30W', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 373, 'type': 'NATIVE', 'title': 'Building real estate valuation models with comparative approach through case-based reasoning', 'authors': 'I. Yeh

In [7]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
        random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=[],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')



================ SINGLE RUN ================
Training TabDDPM...
[0]
6
{'num_classes': 0, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(6)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.4427 Sum: 0.4427
Step 1000/1000 MLoss: 0.0 GLoss: 0.4089 Sum: 0.4089
mlp
Sample timestep    0
Discrete cols: [2]
Num shape:  (1000, 5)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 6/6 [00:00<00:00, 560.97it/s]|
Column Shapes Score: 65.94%

(2/2) Evaluating Column Pair Trends: |██████████| 15/15 [00:00<00:00, 832.20it/s]|
Column Pair Trends Score: 81.01%

Overall Score (Average): 73.48%

TabDDPM: 0.7348


In [8]:
# ForestDiffusion
if 'ForestDiffusion' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training ForestDiffusion...')
        synthetic_forestdiffusion = train_forestdiffusion(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['ForestDiffusion'] = synthetic_forestdiffusion.copy()
        print('ForestDiffusion: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_metadata,
            )
            scores['ForestDiffusion'] = quality.get_score()
            print('ForestDiffusion:', round(scores['ForestDiffusion'], 4))
        else:
            print('ForestDiffusion: trained (quality eval skipped)')
    except Exception as e:
        print('ForestDiffusion Failed (training/sampling):')
        traceback.print_exc()
    if 'ForestDiffusion' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)')


Training ForestDiffusion...
ForestDiffusion: synthesis complete
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 6/6 [00:00<00:00, 119.72it/s]|
Column Shapes Score: 94.31%

(2/2) Evaluating Column Pair Trends: |██████████| 15/15 [00:00<00:00, 241.35it/s]|
Column Pair Trends Score: 99.58%

Overall Score (Average): 96.94%

ForestDiffusion: 0.9694


In [9]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')


Regression evaluation: 10 models, 10 seeds, 6 generators


In [10]:
def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    train_df = align_to_train_schema(train_df, schema_df, label_col)
    test_df = align_to_train_schema(test_df, schema_df, label_col)
    results = []

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if use_holdout:
                # Keep holdout protocol, but bootstrap train/test rows per seed
                # so repeated runs produce meaningful variance estimates.
                rng = np.random.default_rng(seed)
                train_idx = rng.choice(len(X_train), size=len(X_train), replace=True)
                test_idx = rng.choice(len(X_test), size=len(X_test), replace=True)

                X_train = X_train.iloc[train_idx].reset_index(drop=True)
                y_train = y_train.iloc[train_idx].reset_index(drop=True)
                X_test = X_test.iloc[test_idx].reset_index(drop=True)
                y_test = y_test.iloc[test_idx].reset_index(drop=True)
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_train, y_train, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': np.std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': np.std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': np.std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': np.std(mae_scores),
            'R2 (Mean±Std)': f"{np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}",
            'MSE (Mean±Std)': f"{np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}",
            'RMSE (Mean±Std)': f"{np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}",
            'MAE (Mean±Std)': f"{np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [11]:
print('TRTR (Train Real, Test Real) — 80% train / 20% holdout')
trtr_results = evaluate_regression_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=True,
    schema_df=train_real,
)
display(trtr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on 20% holdout)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=True,
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


TRTR (Train Real, Test Real) — 80% train / 20% holdout


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
3,ElasticNet,0.6055 ± 0.0474,69.7461 ± 12.0436,8.3218 ± 0.7030,6.4895 ± 0.6330
1,Ridge,0.6018 ± 0.0459,70.4029 ± 11.8182,8.3623 ± 0.6896,6.5262 ± 0.6026
4,SVR_RBF,0.5727 ± 0.0312,75.1732 ± 7.4784,8.6598 ± 0.4256,6.2732 ± 0.5122
2,Lasso,0.5486 ± 0.0757,79.5963 ± 15.4880,8.8805 ± 0.8562,6.8396 ± 0.8266
0,LinearRegression,0.5430 ± 0.0774,80.5597 ± 15.6744,8.9339 ± 0.8632,6.8835 ± 0.8222
5,KNN,0.5081 ± 0.2027,86.9103 ± 37.1793,9.1273 ± 1.8981,6.7515 ± 1.1603
7,RandomForest,0.4204 ± 0.6399,99.0934 ± 102.6995,9.1770 ± 3.8569,6.3412 ± 1.6844
8,ExtraTrees,0.3820 ± 0.6167,106.3692 ± 99.4856,9.5300 ± 3.9432,6.4319 ± 1.7555
9,GradientBoost,0.3657 ± 0.8038,108.1949 ± 129.5118,9.3382 ± 4.5818,6.3784 ± 1.8860
6,DecisionTree,0.3485 ± 0.4625,114.5330 ± 82.8031,10.1700 ± 3.3324,7.0632 ± 1.3893


Skipping CTGAN - no synthetic dataset
Skipping CopulaGAN - no synthetic dataset
Skipping TVAE - no synthetic dataset
Skipping GaussianCopula - no synthetic dataset
TabDDPM - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
8,ExtraTrees,0.5642 ± 0.0935,73.7834 ± 15.7255,8.5363 ± 0.9563,6.2054 ± 0.6564
4,SVR_RBF,0.5230 ± 0.0585,81.5999 ± 15.9322,8.9882 ± 0.9016,6.9521 ± 0.7915
9,GradientBoost,0.4857 ± 0.1220,86.4373 ± 18.0007,9.2434 ± 0.9987,6.7316 ± 0.4997
7,RandomForest,0.4137 ± 0.1370,99.7842 ± 24.8081,9.9009 ± 1.3254,6.9703 ± 0.8448
5,KNN,0.3927 ± 0.1231,104.9840 ± 30.0189,10.1371 ± 1.4910,7.2578 ± 1.0212
2,Lasso,0.1363 ± 0.1990,144.2122 ± 23.7715,11.9672 ± 0.9997,9.4088 ± 0.6220
0,LinearRegression,0.1337 ± 0.1996,144.6609 ± 23.9638,11.9854 ± 1.0061,9.4149 ± 0.6256
3,ElasticNet,0.0952 ± 0.2169,150.1754 ± 20.1153,12.2266 ± 0.8284,10.1796 ± 0.6883
1,Ridge,0.0298 ± 0.2273,161.2277 ± 22.0536,12.6671 ± 0.8784,10.6543 ± 0.7448
6,DecisionTree,0.0234 ± 0.2344,165.5146 ± 38.1162,12.7750 ± 1.5211,8.9943 ± 1.1253


ForestDiffusion - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
8,ExtraTrees,0.7305 ± 0.0468,45.2948 ± 6.0260,6.7145 ± 0.4586,4.9382 ± 0.3614
9,GradientBoost,0.7182 ± 0.0694,47.2371 ± 10.0755,6.8357 ± 0.7140,5.1234 ± 0.6407
7,RandomForest,0.6896 ± 0.0932,51.4502 ± 12.5382,7.1237 ± 0.8388,5.2071 ± 0.5779
4,SVR_RBF,0.6218 ± 0.0466,64.8733 ± 13.6906,8.0087 ± 0.8571,5.8118 ± 0.5496
3,ElasticNet,0.5777 ± 0.0496,71.3791 ± 8.6455,8.4330 ± 0.5139,6.8244 ± 0.4380
1,Ridge,0.5746 ± 0.0511,71.9019 ± 8.8323,8.4633 ± 0.5231,6.8531 ± 0.4466
2,Lasso,0.5553 ± 0.0513,75.0193 ± 7.6674,8.6498 ± 0.4466,6.8947 ± 0.3888
0,LinearRegression,0.5518 ± 0.0522,75.5967 ± 7.7438,8.6830 ± 0.4495,6.9087 ± 0.3971
6,DecisionTree,0.4831 ± 0.1933,85.6967 ± 27.7677,9.1458 ± 1.4323,6.4569 ± 0.9855
5,KNN,0.2182 ± 0.4116,129.0384 ± 63.8844,11.0094 ± 2.7985,7.0670 ± 1.4807


,Synthetic_Model,R2_Drop,MSE_Increase,RMSE_Increase,MAE_Increase
0,ForestDiffusion,-0.082454,-17.309125,-0.743373,-0.389285
1,TabDDPM,0.209839,32.180082,1.792642,1.679113


In [12]:
# Create quality metrics DataFrame
quality_df = pd.DataFrame({
    'Generator': list(scores.keys()),
    'Quality_Score': list(scores.values())
})

output_file = 'TRTR_TSTR_results_real_estate.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')


Results saved to: TRTR_TSTR_results_real_estate.xlsx
